# SmartPump AI — Production Data Preprocessing & Feature Engineering

**Module:** `notebooks/02_preprocessing.ipynb`  
**Role:** Senior Machine Learning Engineer, Data Engineer, & Predictive Maintenance Specialist  
**Target:** Multiclass Classification of Internal Hydraulic Pump Leakage (`0: Normal`, `1: Warning`, `2: Fault`)  
**Dataset:** UCI Machine Learning Repository — *Condition Monitoring of Hydraulic Systems* (ID 447)  
**Output Datasets:** `data/processed/stable_cycle_features.csv`, `data/processed/full_cycle_features.csv`, metadata catalogs  

---

## Executive Summary & Engineering Architecture

This notebook executes the end-to-end production data preprocessing and feature engineering pipeline for **SmartPump AI**. The primary objective is to transform raw, multi-rate sensor time-series (17 channels, 1 Hz to 100 Hz, 43,680 time samples per cycle) into a structured, cycle-level feature matrix suitable for supervised machine learning.

### Key Pipeline Guarantees:
1. **Observation Unit Integrity:** The fundamental unit of observation is one 60-second operating cycle. Individual time samples are never split across train/test.
2. **Zero Missingness & Infinite Values:** Handled with strict numeric validation, preserving true physical zero readings (notably accumulator gauge pressure `PS4`).
3. **Dual Population Strategy:** Generates both a primary **Stable-Cycle Dataset** (1,449 cycles, nearly balanced 1:1:1) and a secondary **Full-Cycle Dataset** (2,205 cycles) to evaluate operational drift and transient dynamics.
4. **Strict Leakage Prevention:** Feature extraction is self-contained per cycle. Splitting follows a 70% / 15% / 15% stratified cycle partition, with scalers fit strictly on training data.

---

## Table of Contents
1. [Setup and Environment Configuration](#1.-Setup-and-Environment-Configuration)
2. [Loading Condition Profile & Validating Ground Truth](#2.-Loading-Condition-Profile-&-Validating-Ground-Truth)
3. [Sensor Data Loading & Multi-Rate Architecture](#3.-Sensor-Data-Loading-&-Multi-Rate-Architecture)
4. [Cycle-Level Feature Engineering (Statistical, Distribution, Temporal)](#4.-Cycle-Level-Feature-Engineering)
5. [Population Analysis: Stable vs. Full Operating Cycles](#5.-Population-Analysis:-Stable-vs.-Full-Operating-Cycles)
6. [Feature Selection & Priority Sensor Discriminability](#6.-Feature-Selection-&-Priority-Sensor-Discriminability)
7. [Exporting Production Processed Datasets](#7.-Exporting-Production-Processed-Datasets)
8. [Leakage-Safe Data Splitting & Feature Scaling Demonstration](#8.-Leakage-Safe-Data-Splitting-&-Feature-Scaling-Demonstration)
9. [Verification & Pipeline Sign-off](#9.-Verification-&-Pipeline-Sign-off)


## 1. Setup and Environment Configuration


In [1]:
import os
import sys
from pathlib import Path

# Configure writable cache and non-blocking headless plotting backend
os.environ["MPLCONFIGDIR"] = os.environ.get("TEMP", "/tmp")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Add src directory to path
PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import preprocessing as prep
import feature_engineering as fe

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

# Visual configuration
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

print("Environment initialized successfully.")
print(f"Project root:  {PROJECT_ROOT}")
print(f"Raw data dir:  {RAW_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")


Environment initialized successfully.
Project root:  C:\Users\hp_pa\OneDrive\Desktop\SmartPump-AI
Raw data dir:  C:\Users\hp_pa\OneDrive\Desktop\SmartPump-AI\data\raw
Processed dir: C:\Users\hp_pa\OneDrive\Desktop\SmartPump-AI\data\processed


## 2. Loading Condition Profile & Validating Ground Truth

The `profile.txt` ground-truth file provides component status across all 2,205 operating cycles:
- `col 0`: Cooler condition (`3`: failure, `20`: reduced, `100`: full)
- `col 1`: Valve condition (`73`: failure, `80`: severe lag, `90`: small lag, `100`: optimal)
- `col 2`: **Internal Pump Leakage (Target)**: `0 = Normal`, `1 = Warning`, `2 = Fault`
- `col 3`: Hydraulic accumulator pre-charge pressure (`90`, `100`, `115`, `130` bar)
- `col 4`: Operational stability flag (`0 = stable`, `1 = unstable/transient`)


In [2]:
# Load validated ground-truth profile
profile = prep.load_profile(RAW_DIR)
full_metadata = prep.create_cycle_metadata(profile)

print(f"Profile successfully loaded. Total cycles: {len(profile)}")
print("Sample of cycle-level operational metadata:")
print(full_metadata.head(5).to_string())

# Describe overall dataset
prep.describe_dataset(profile)


Profile successfully loaded. Total cycles: 2205
Sample of cycle-level operational metadata:
   cycle_id  target target_name  cooler  valve  accumulator  stable_flag  is_stable             cooler_desc         valve_desc              accumulator_desc
0         0       0      Normal       3    100          130            1      False  close to total failure  optimal switching  130 bar (optimal pre-charge)
1         1       0      Normal       3    100          130            1      False  close to total failure  optimal switching  130 bar (optimal pre-charge)
2         2       0      Normal       3    100          130            1      False  close to total failure  optimal switching  130 bar (optimal pre-charge)
3         3       0      Normal       3    100          130            1      False  close to total failure  optimal switching  130 bar (optimal pre-charge)
4         4       0      Normal       3    100          130            1      False  close to total failure  optimal switch

## 3. Sensor Data Loading & Multi-Rate Architecture

The test rig records 17 sensor channels with three distinct sampling rates:
- **100 Hz** (6,000 samples/cycle): `PS1`–`PS6` (hydraulic pressures) and `EPS1` (motor power)
- **10 Hz** (600 samples/cycle): `FS1` (primary flow) and `FS2` (cooling flow)
- **1 Hz** (60 samples/cycle): `TS1`–`TS4` (temperatures), `VS1` (vibration amplitude), `CE`, `CP`, `SE` (virtual metrics)

We validate file integrity, ensuring 0 missing values and 0 infinite values across all 43,680 measurements per cycle.


In [3]:
# Load all 17 raw sensor channels
sensors = prep.load_all_sensors(RAW_DIR, verbose=True)

# Run comprehensive validation audit
audit_report = prep.validate_raw_data(profile, sensors)
print("\n--- Raw Data Validation Audit ---")
print(f"Total Cycles Audited:  {audit_report['total_cycles']}")
print(f"Sensors Audited:       {audit_report['sensor_count']}")
print(f"Shapes Valid:          {audit_report['shapes_valid']}")
print(f"Missing Values (NaN):  {audit_report['missing_values']}")
print(f"Infinite Values (Inf): {audit_report['infinite_values']}")
assert audit_report["missing_values"] == 0 and audit_report["infinite_values"] == 0
print("Data integrity check PASSED.")


[preprocessing] Loaded PS1.txt  | Shape: 2205 cycles × 6000 samples (100 Hz)
[preprocessing] Loaded PS2.txt  | Shape: 2205 cycles × 6000 samples (100 Hz)
[preprocessing] Loaded PS3.txt  | Shape: 2205 cycles × 6000 samples (100 Hz)
[preprocessing] Loaded PS4.txt  | Shape: 2205 cycles × 6000 samples (100 Hz)
[preprocessing] Loaded PS5.txt  | Shape: 2205 cycles × 6000 samples (100 Hz)
[preprocessing] Loaded PS6.txt  | Shape: 2205 cycles × 6000 samples (100 Hz)
[preprocessing] Loaded EPS1.txt | Shape: 2205 cycles × 6000 samples (100 Hz)
[preprocessing] Loaded FS1.txt  | Shape: 2205 cycles × 600 samples ( 10 Hz)
[preprocessing] Loaded FS2.txt  | Shape: 2205 cycles × 600 samples ( 10 Hz)
[preprocessing] Loaded TS1.txt  | Shape: 2205 cycles × 60 samples (  1 Hz)
[preprocessing] Loaded TS2.txt  | Shape: 2205 cycles × 60 samples (  1 Hz)
[preprocessing] Loaded TS3.txt  | Shape: 2205 cycles × 60 samples (  1 Hz)
[preprocessing] Loaded TS4.txt  | Shape: 2205 cycles × 60 samples (  1 Hz)
[preproce

## 4. Cycle-Level Feature Engineering

For each sensor channel, we extract 15 domain-justified summary features per operating cycle:
1. **Statistical Central Tendency:** `mean`, `median`
2. **Statistical Dispersion:** `std`, `range`, `iqr`
3. **Statistical Extremes & Percentiles:** `min`, `max`, `q25`, `q75`
4. **Signal Energy:** `rms` (Root Mean Square)
5. **Distribution Shape:** `skewness`, `kurtosis` (with zero-variance safety guards)
6. **Temporal Dynamics:** `slope` (closed-form linear regression rate), `delta` (first-to-last net change), `variability` (mean absolute consecutive difference)

Total feature space: 17 sensors × 15 statistics = **255 engineered features per cycle**.


In [4]:
# Extract all 255 features across all 2,205 operating cycles
print("Extracting cycle-level features across all 17 sensors...")
full_features = fe.build_feature_matrix(sensors, statistics=fe.ALL_STATISTICS, verbose=True)

print(f"\nFull feature matrix assembled: {full_features.shape[0]} cycles × {full_features.shape[1]} features")

# Verify no NaNs or Infs generated
nan_total = int(full_features.isna().sum().sum())
inf_total = int(np.isinf(full_features.to_numpy()).sum())
print(f"Total NaN features: {nan_total}")
print(f"Total Inf features: {inf_total}")
assert nan_total == 0 and inf_total == 0, "Feature matrix contains invalid numerical values!"
print("Feature matrix numerical integrity: CLEAN (0 NaN, 0 Inf).")


Extracting cycle-level features across all 17 sensors...
[feature_engineering] PS1   (PS1.txt ) -> Extracted 15 features across 2205 cycles.
[feature_engineering] PS2   (PS2.txt ) -> Extracted 15 features across 2205 cycles.
[feature_engineering] PS3   (PS3.txt ) -> Extracted 15 features across 2205 cycles.
[feature_engineering] PS4   (PS4.txt ) -> Extracted 15 features across 2205 cycles.
[feature_engineering] PS5   (PS5.txt ) -> Extracted 15 features across 2205 cycles.
[feature_engineering] PS6   (PS6.txt ) -> Extracted 15 features across 2205 cycles.
[feature_engineering] EPS1  (EPS1.txt) -> Extracted 15 features across 2205 cycles.
[feature_engineering] FS1   (FS1.txt ) -> Extracted 15 features across 2205 cycles.
[feature_engineering] FS2   (FS2.txt ) -> Extracted 15 features across 2205 cycles.
[feature_engineering] TS1   (TS1.txt ) -> Extracted 15 features across 2205 cycles.
[feature_engineering] TS2   (TS2.txt ) -> Extracted 15 features across 2205 cycles.
[feature_engineerin

## 5. Population Analysis: Stable vs. Full Operating Cycles

We construct two distinct datasets:
1. **Stable-Cycle Dataset (Primary):** Filtered for `stable_flag == 0` (1,449 cycles). Nearly balanced: 489 Normal, 480 Warning, 480 Fault.
2. **Full-Cycle Dataset (Secondary):** Contains all 2,205 cycles (1,221 Normal, 492 Warning, 492 Fault) to evaluate operational transients and robustness.


In [5]:
# Filter stable cycles
stable_metadata, _ = prep.filter_stable_cycles(full_metadata)
stable_features = full_features.loc[stable_metadata["cycle_id"]].copy()

print(f"Stable features shape: {stable_features.shape}")
print(f"Stable metadata shape: {stable_metadata.shape}")

# Visual comparison of population distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stable population
stable_counts = stable_metadata["target_name"].value_counts()
axes[0].bar(stable_counts.index, stable_counts.values, color=["#2ecc71", "#f39c12", "#e74c3c"])
axes[0].set_title(f"Primary: Stable Operating Cycles (N = {len(stable_metadata)})")
axes[0].set_ylabel("Number of Cycles")
for i, v in enumerate(stable_counts.values):
    axes[0].text(i, v + 10, f"{v} ({100*v/len(stable_metadata):.1f}%)", ha="center", fontweight="bold")

# Full population
full_counts = full_metadata["target_name"].value_counts()
axes[1].bar(full_counts.index, full_counts.values, color=["#2ecc71", "#f39c12", "#e74c3c"])
axes[1].set_title(f"Secondary: All Operating Cycles (N = {len(full_metadata)})")
axes[1].set_ylabel("Number of Cycles")
for i, v in enumerate(full_counts.values):
    axes[1].text(i, v + 20, f"{v} ({100*v/len(full_metadata):.1f}%)", ha="center", fontweight="bold")

plt.tight_layout()
fig_path = RESULTS_DIR / "eda" / "visualizations" / "population_comparison.png"
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150)
plt.close(fig)
print(f"Saved population comparison figure: {fig_path}")


[preprocessing] Filtered 756 unstable cycles (stable_flag == 1). Retained 1449 stable cycles (stable_flag == 0).
Stable features shape: (1449, 255)
Stable metadata shape: (1449, 11)
Saved population comparison figure: C:\Users\hp_pa\OneDrive\Desktop\SmartPump-AI\results\eda\visualizations\population_comparison.png


## 6. Feature Selection & Priority Sensor Discriminability

We examine the priority sensor channels identified during EDA:
- `FS1` (Primary volume flow rate): Direct hydrodynamic symptom of internal slip/leakage
- `SE` (Efficiency factor): Overall mechanical-hydraulic transmission efficiency
- `EPS1` (Motor power): Electrical drive power compensation for leakage losses
- `PS1`, `PS2`, `PS3` (Hydraulic pressures): Pressure dynamics across high/low pressure circuits

We also check for **zero-variance features** (e.g. `min` values that always hit 0 during cylinder return).


In [6]:
# Identify constant/zero-variance features across stable cycles
feature_stds = stable_features.std()
zero_var_features = feature_stds[feature_stds < 1e-10].index.tolist()
print(f"Zero-variance features identified ({len(zero_var_features)}): {zero_var_features}")
print("Explanation: These represent cycle minimums (e.g. PS2_min, FS1_min) that consistently reach 0.0 bar/l/min during valve switching/tank return.")

# Calculate correlations of priority sensor features with target
priority_cols = [c for c in stable_features.columns if any(c.startswith(f"{s}_") for s in fe.PRIORITY_SENSORS)]
correlations = stable_features[priority_cols].apply(lambda col: col.corr(stable_metadata["target"])).sort_values(ascending=False)

print("\nTop 10 Positively Correlated Features with Pump Leakage:")
print(correlations.head(10).to_string())

print("\nTop 10 Negatively Correlated Features with Pump Leakage:")
print(correlations.tail(10).to_string())

# Visualize feature distributions for top discriminative features
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.boxplot(x=stable_metadata["target_name"], y=stable_features["FS1_mean"], ax=axes[0], palette="Set2")
axes[0].set_title("Primary Flow: FS1 Mean [l/min]")
axes[0].set_xlabel("Leakage Condition")

sns.boxplot(x=stable_metadata["target_name"], y=stable_features["SE_mean"], ax=axes[1], palette="Set2")
axes[1].set_title("Hydraulic Efficiency: SE Mean [%]")
axes[1].set_xlabel("Leakage Condition")

sns.boxplot(x=stable_metadata["target_name"], y=stable_features["EPS1_mean"], ax=axes[2], palette="Set2")
axes[2].set_title("Motor Drive Power: EPS1 Mean [W]")
axes[2].set_xlabel("Leakage Condition")

plt.tight_layout()
fig_dist_path = RESULTS_DIR / "eda" / "visualizations" / "priority_feature_distributions.png"
plt.savefig(fig_dist_path, dpi=150)
plt.close(fig)
print(f"Saved priority feature distributions figure: {fig_dist_path}")


Zero-variance features identified (3): ['PS2_min', 'PS3_min', 'SE_min']
Explanation: These represent cycle minimums (e.g. PS2_min, FS1_min) that consistently reach 0.0 bar/l/min during valve switching/tank return.

Top 10 Positively Correlated Features with Pump Leakage:
EPS1_rms       0.383322
EPS1_mean      0.382644
EPS1_q75       0.353601
EPS1_median    0.322111
FS1_skew       0.263944
PS2_q75        0.241232
PS2_iqr        0.238478
PS2_std        0.238241
PS2_skew       0.236779
PS2_rms        0.235675

Top 10 Negatively Correlated Features with Pump Leakage:
SE_q25     -0.309426
SE_slope   -0.317331
FS1_mean   -0.320312
SE_mean    -0.359172
FS1_rms    -0.359232
SE_q75     -0.364596
SE_rms     -0.410987
PS2_min          NaN
PS3_min          NaN
SE_min           NaN
Saved priority feature distributions figure: C:\Users\hp_pa\OneDrive\Desktop\SmartPump-AI\results\eda\visualizations\priority_feature_distributions.png


C:\Users\hp_pa\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\hp_pa\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
<string>:20: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

<string>:24: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

<string>:28: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.



## 7. Exporting Production Processed Datasets

We persist the standardized processed datasets in `data/processed/`:
1. `stable_cycle_features.csv`: 1,449 cycles × 255 features + `cycle_id` + `target`
2. `full_cycle_features.csv`: 2,205 cycles × 255 features + `cycle_id` + `target`
3. `stable_metadata.csv`: 1,449 rows operational metadata
4. `full_metadata.csv`: 2,205 rows operational metadata
5. `feature_metadata.csv`: 255 rows feature documentation catalog
6. `features_per_cycle.csv`: Backward-compatible export for existing model scripts


In [7]:
# Generate feature metadata catalog
feature_metadata = fe.generate_feature_metadata(list(full_features.columns))
print(f"Generated feature catalog: {len(feature_metadata)} entries")
print(feature_metadata.head(5).to_string())

# Persist all datasets to data/processed/
saved_paths = fe.save_processed_datasets(
    stable_features=stable_features,
    full_features=full_features,
    stable_metadata=stable_metadata,
    full_metadata=full_metadata,
    processed_dir=PROCESSED_DIR,
    feature_metadata=feature_metadata,
)

print("\n--- Saved Dataset Files ---")
for key, path in saved_paths.items():
    print(f"{key:25s}: {path.name:28s} ({path.stat().st_size / (1024*1024):.2f} MB)")


Generated feature catalog: 255 entries
  feature_name sensor statistic                      category unit sampling_rate                         sensor_subsystem  is_priority_sensor                                                                                        description
0     PS1_mean    PS1      mean  statistical_central_tendency  bar        100 Hz  Primary circuit pressure at pump outlet                True           Primary circuit pressure at pump outlet - Arithmetic mean of signal over 60-second cycle
1      PS1_std    PS1       std        statistical_dispersion  bar        100 Hz  Primary circuit pressure at pump outlet                True  Primary circuit pressure at pump outlet - Sample standard deviation (ddof=1) over 60-second cycle
2      PS1_min    PS1       min          statistical_extremes  bar        100 Hz  Primary circuit pressure at pump outlet                True                      Primary circuit pressure at pump outlet - Minimum value observed during cyc

## 8. Leakage-Safe Data Splitting & Feature Scaling Demonstration

We demonstrate a production-grade data splitting workflow:
- **70% Training / 15% Validation / 15% Testing**
- **Unit of Split:** Operating cycle (`cycle_id`)
- **Stratified** by target label (`Normal`, `Warning`, `Fault`)
- **Zero Cycle Overlap** verified via set assertions
- **Feature Scaling:** `StandardScaler` is fitted **strictly on X_train** and applied to validation and test sets.


In [8]:
# Load clean feature matrix, target, and metadata
X_stable, y_stable, meta_stable = fe.load_processed_dataset(PROCESSED_DIR, dataset_type="stable")

# Split data into 70% Train, 15% Val, 15% Test
splits = prep.split_cycle_data(
    X_stable, y_stable, metadata=meta_stable,
    train_size=0.70, val_size=0.15, test_size=0.15,
    random_state=42, stratify=True,
)

X_train, X_val, X_test = splits["X_train"], splits["X_val"], splits["X_test"]
y_train, y_val, y_test = splits["y_train"], splits["y_val"], splits["y_test"]

print(f"X_train shape: {X_train.shape} (70.0%)")
print(f"X_val shape:   {X_val.shape} (15.0%)")
print(f"X_test shape:  {X_test.shape} (15.0%)")

# Verify zero cycle overlap
train_cycles = set(splits["indices"]["train"])
val_cycles = set(splits["indices"]["val"])
test_cycles = set(splits["indices"]["test"])

assert len(train_cycles.intersection(val_cycles)) == 0
assert len(train_cycles.intersection(test_cycles)) == 0
assert len(val_cycles.intersection(test_cycles)) == 0
print("Verified: Zero cycle overlap between Train, Validation, and Test partitions.")

# Display class distribution across splits
split_dist = pd.DataFrame({
    "Train Count": y_train.value_counts().sort_index(),
    "Train %": y_train.value_counts(normalize=True).sort_index() * 100,
    "Val Count": y_val.value_counts().sort_index(),
    "Val %": y_val.value_counts(normalize=True).sort_index() * 100,
    "Test Count": y_test.value_counts().sort_index(),
    "Test %": y_test.value_counts(normalize=True).sort_index() * 100,
})
split_dist.index = ["0: Normal", "1: Warning", "2: Fault"]
print("\nClass distribution across partitions:")
print(split_dist.to_string())

# Demonstrate strict fit-on-train feature scaling
X_train_scaled, X_val_scaled, X_test_scaled, scaler = prep.fit_transform_scaler(X_train, X_val, X_test)
print("\n--- Feature Scaling Leakage Audit ---")
print(f"Fitted Scaler Means:     {scaler.mean_.shape[0]} features")
print(f"X_train Scaled Mean Max: {abs(X_train_scaled.mean()).max():.2e} (should be ~0)")
print("Feature scaling completed without data leakage.")


X_train shape: (1014, 255) (70.0%)
X_val shape:   (217, 255) (15.0%)
X_test shape:  (218, 255) (15.0%)
Verified: Zero cycle overlap between Train, Validation, and Test partitions.

Class distribution across partitions:
            Train Count    Train %  Val Count      Val %  Test Count     Test %
0: Normal           342  33.727811         73  33.640553          74  33.944954
1: Warning          336  33.136095         72  33.179724          72  33.027523
2: Fault            336  33.136095         72  33.179724          72  33.027523

--- Feature Scaling Leakage Audit ---
Fitted Scaler Means:     255 features
X_train Scaled Mean Max: 1.41e-14 (should be ~0)
Feature scaling completed without data leakage.


## 9. Verification & Pipeline Sign-off

All pipeline checks and data contracts have been verified:
- Cycle counts: exactly 2,205 full cycles and 1,449 stable cycles
- Feature count: 255 engineered features per cycle (17 sensors × 15 statistics)
- Target alignment: 489 Normal, 480 Warning, 480 Fault (stable population)
- Data integrity: 0 NaNs, 0 Infs across all feature matrices
- Leakage safety: cycle-level splitting, zero partition overlap, strictly fit on train


In [9]:
print("===========================================================")
print("  SmartPump AI - Step 3 Preprocessing & Engineering COMPLETE")
print("===========================================================")
print(f"Stable Dataset: {X_stable.shape[0]} cycles × {X_stable.shape[1]} features")
print(f"Full Dataset:   {full_features.shape[0]} cycles × {full_features.shape[1]} features")
print(f"Target Classes: Normal (0), Warning (1), Fault (2)")
print(f"Status: READY FOR STEP 4 MODEL TRAINING")
print("===========================================================")


  SmartPump AI - Step 3 Preprocessing & Engineering COMPLETE
Stable Dataset: 1449 cycles × 255 features
Full Dataset:   2205 cycles × 255 features
Target Classes: Normal (0), Warning (1), Fault (2)
Status: READY FOR STEP 4 MODEL TRAINING
